In [ ]:

# CHINESE WHISPERS USING BiLSTM
# Step 1: Import the required libraries

import tensorflow as tf
import numpy as np
import random
import re

print("Chinese Whispers Project")
print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")


Chinese Whispers Project
TensorFlow version: 2.21.0
Libraries imported successfully!


In [ ]:

# STEP 3: Prepare training sentences

sentences = [
    "The little boy carried a blue umbrella",
    "The girl is reading a book",
    "A dog is playing in the garden",
    "The teacher writes on the board",
    "My friend went to the market",
    "The cat is sleeping on the sofa",
    "Birds are flying over the green trees",
    "I like to learn new things",
    "The children are playing outside",
    "Please bring me a glass of water",
    "The sun rises in the east",
    "She is cooking dinner for her family",
    "We are going to school today",
    "The train arrived at the station",
    "He bought a red shirt yesterday",
    "The farmer works in the field",
    "My mother made a delicious cake",
    "The students are studying together",
    "A small rabbit jumped over the fence",
    "We watched a movie last night"
]

print("Training sentences prepared:", len(sentences))
print("\nExample sentences:")
for i, sentence in enumerate(sentences[:5], start=1):
    print(f"{i}. {sentence}")


Training sentences prepared: 20

Example sentences:
1. The little boy carried a blue umbrella
2. The girl is reading a book
3. A dog is playing in the garden
4. The teacher writes on the board
5. My friend went to the market


In [ ]:

# STEP 4: Generate corrupted training examples

import random
import re

random.seed(42)

def corrupt_sentence(sentence, error_rate=0.20):
    words = sentence.split()
    result = words.copy()

    for i in range(len(result)):
        if random.random() < error_rate:
            operation = random.choice(["replace", "delete", "swap"])

            if operation == "replace":
                result[i] = random.choice(
                    ["small", "big", "happy", "new", "green", "red"]
                )

            elif operation == "delete":
                result[i] = ""

            elif operation == "swap" and i < len(result) - 1:
                result[i], result[i + 1] = result[i + 1], result[i]

    return " ".join(word for word in result if word)


# Generate several examples from each sentence
training_inputs = []
training_targets = []

for sentence in sentences:
    for _ in range(10):
        noisy = corrupt_sentence(sentence)
        training_inputs.append(noisy)
        training_targets.append(sentence)

print("Training pairs:", len(training_inputs))
print("\nExample training pairs:")

for i in range(5):
    print("Noisy input :", training_inputs[i])
    print("Target      :", training_targets[i])
    print()


Training pairs: 200

Example training pairs:
Noisy input : The boy red a blue umbrella
Target      : The little boy carried a blue umbrella

Noisy input : big little boy carried a blue umbrella
Target      : The little boy carried a blue umbrella

Noisy input : The little boy carried blue
Target      : The little boy carried a blue umbrella

Noisy input : boy carried a blue umbrella
Target      : The little boy carried a blue umbrella

Noisy input : The little boy carried a blue umbrella
Target      : The little boy carried a blue umbrella



In [ ]:

# STEP 5: Tokenize and pad the training data

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create a vocabulary from noisy inputs and target sentences
tokenizer = Tokenizer(
    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n',
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(training_inputs + training_targets)

# Convert sentences into sequences of integers
input_sequences = tokenizer.texts_to_sequences(training_inputs)
target_sequences = tokenizer.texts_to_sequences(training_targets)

# Determine the maximum sequence length
max_length = max(
    max(len(seq) for seq in input_sequences),
    max(len(seq) for seq in target_sequences)
)

# Pad sequences so they all have equal length
X = pad_sequences(
    input_sequences,
    maxlen=max_length,
    padding="post"
)

Y = pad_sequences(
    target_sequences,
    maxlen=max_length,
    padding="post"
)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)
print("Maximum sentence length:", max_length)
print("Input shape:", X.shape)
print("Target shape:", Y.shape)

print("\nExample sentence:", training_inputs[0])
print("Numerical sequence:", X[0])


Vocabulary size: 90
Maximum sentence length: 7
Input shape: (200, 7)
Target shape: (200, 7)

Example sentence: The boy red a blue umbrella
Numerical sequence: [ 2 17 13  3 18 33  0]


In [ ]:

# STEP 6: Build the BiLSTM neural network

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    TimeDistributed,
    Dense
)

# Build the model
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(64, return_sequences=True)
    ),

    TimeDistributed(
        Dense(vocab_size, activation="softmax")
    )
])

# Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

# Display the architecture
model.build(input_shape=(None, max_length))
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 7, 64)          │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 7, 128)         │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 7, 90)          │        11,610 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,418 (325.85 KB)

 Trainable params: 83,418 (325.85 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

# STEP 7: Train the BiLSTM model

import numpy as np

# Convert targets to the shape expected by the model
Y_train = np.expand_dims(Y, axis=-1)

# Train the model
history = model.fit(
    X,
    Y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

print("\nTraining completed!")


Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - loss: 4.4821 - sparse_categorical_accuracy: 0.1551 - val_loss: 4.4914 - val_sparse_categorical_accuracy: 0.0922
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 4.4209 - sparse_categorical_accuracy: 0.1855 - val_loss: 4.4762 - val_sparse_categorical_accuracy: 0.0922
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 4.2619 - sparse_categorical_accuracy: 0.1709 - val_loss: 4.4448 - val_sparse_categorical_accuracy: 0.0829
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.8578 - sparse_categorical_accuracy: 0.1688 - val_loss: 4.4958 - val_sparse_categorical_accuracy: 0.0829
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 3.6140 - sparse_categorical_accuracy: 0.1730 - val_loss: 4.5562 - val_sparse_categorical_accuracy: 0.1014
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 3.4129 - sparse_categorical_accuracy: 0.2201 - val_loss: 4.6360 - val_sparse_categorical_accuracy: 0.1429
Epoc

In [ ]:

# Display the final training metrics

print("Final training loss:",
      round(history.history["loss"][-1], 4))

print("Final training accuracy:",
      round(history.history["sparse_categorical_accuracy"][-1], 4))

print("Final validation loss:",
      round(history.history["val_loss"][-1], 4))

print("Final validation accuracy:",
      round(history.history["val_sparse_categorical_accuracy"][-1], 4))


Final training loss: 0.0421
Final training accuracy: 1.0
Final validation loss: 7.9816
Final validation accuracy: 0.1659


In [ ]:

def predict_sentence(sentence):
    # Convert the sentence into numbers
    sequence = tokenizer.texts_to_sequences([sentence])

    # Make the sequence the same length as the training input
    padded = tf.keras.preprocessing.sequence.pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    # Ask the trained BiLSTM to predict the output
    predictions = model.predict(padded, verbose=0)

    # Choose the most likely word at each position
    predicted_ids = np.argmax(predictions[0], axis=-1)

    # Convert predicted numbers back into words
    predicted_words = []

    for word_id in predicted_ids:
        if word_id == 0:
            continue

        word = tokenizer.index_word.get(word_id)

        if word:
            predicted_words.append(word)

    return " ".join(predicted_words)


# Test the trained model
noisy_sentence = "the cat sat on mat"

print("Noisy input :", noisy_sentence)
print("BiLSTM output:", predict_sentence(noisy_sentence))


Noisy input : the cat sat on mat
BiLSTM output: the cat writes on the things things


In [ ]:

import random
import re

random.seed(42)

def whisper_change(sentence):
    words = sentence.split()

    if not words:
        return sentence

    # Pick one small change to simulate mishearing
    change = random.choice(["replace", "remove", "swap"])

    if change == "replace":
        index = random.randrange(len(words))
        alternatives = {
            "cat": "cap",
            "sat": "sang",
            "mat": "map",
            "dog": "duck",
            "the": "a",
            "on": "near",
            "sun": "son",
            "park": "part"
        }
        word = words[index].lower()
        words[index] = alternatives.get(word, "unknown")

    elif change == "remove" and len(words) > 2:
        index = random.randrange(len(words))
        words.pop(index)

    elif change == "swap" and len(words) > 1:
        index = random.randrange(len(words) - 1)
        words[index], words[index + 1] = (
            words[index + 1], words[index]
        )

    return " ".join(words)


original_message = "The cat sat on the mat"
current_message = original_message

print("Original message:", original_message)
print("-" * 50)

for player in range(1, 9):
    current_message = whisper_change(current_message)
    print(f"Player {player}: {current_message}")


Original message: The cat sat on the mat
--------------------------------------------------
Player 1: cat The sat on the mat
Player 2: cat The sat on the map
Player 3: cat sat on the map
Player 4: cat sang on the map
Player 5: sang cat on the map
Player 6: cat sang on the map
Player 7: cat sang on map the
Player 8: cap sang on map the


In [ ]:

from IPython.display import display, Javascript
from google.colab import output

def record_and_transcribe():
    display(Javascript("""
    (async () => {
      const recognition = new (
        window.SpeechRecognition ||
        window.webkitSpeechRecognition
      )();

      recognition.lang = 'en-US';
      recognition.interimResults = false;
      recognition.maxAlternatives = 1;

      recognition.onresult = (event) => {
        const text = event.results[0][0].transcript;
        google.colab.kernel.invokeFunction(
          'notebook.setTranscript', [text], {}
        );
      };

      recognition.onerror = (event) => {
        google.colab.kernel.invokeFunction(
          'notebook.setTranscript',
          ['ERROR: ' + event.error], {}
        );
      };

      recognition.start();
      alert('Speak your sentence after closing this message.');
    })();
    """))

    return output.eval_js("""
      new Promise((resolve) => {
        window.setTranscriptResult = resolve;
      })
    """)


In [ ]:

from IPython.display import display, Javascript
from google.colab import output
import time

transcript_result = {"text": ""}

def save_transcript(text):
    transcript_result["text"] = text

output.register_callback(
    "notebook.save_transcript", save_transcript
)

display(Javascript("""
(async () => {
  const SpeechRecognition =
    window.SpeechRecognition || window.webkitSpeechRecognition;

  if (!SpeechRecognition) {
    alert("Speech recognition is not supported in this browser.");
    return;
  }

  const recognition = new SpeechRecognition();
  recognition.lang = "en-US";
  recognition.interimResults = false;
  recognition.maxAlternatives = 1;

  recognition.onresult = (event) => {
    const text = event.results[0][0].transcript;
    google.colab.kernel.invokeFunction(
      "notebook.save_transcript", [text], {}
    );
    document.getElementById("mic-status").innerText =
      "Recognized: " + text;
  };

  recognition.onerror = (event) => {
    document.getElementById("mic-status").innerText =
      "Microphone error: " + event.error;
  };

  recognition.start();
  document.getElementById("mic-status").innerText =
    "Listening... speak now.";
})();
"""))

display(Javascript("""
if (!document.getElementById("mic-status")) {
  const el = document.createElement("p");
  el.id = "mic-status";
  el.innerText = "Preparing microphone...";
  document.body.appendChild(el);
}
"""))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

from IPython.display import display, Javascript
from google.colab import output
import time

transcript_result = {"text": ""}

def save_transcript(text):
    transcript_result["text"] = text

output.register_callback(
    "notebook.save_transcript", save_transcript
)

display(Javascript("""
(async () => {
  const SpeechRecognition =
    window.SpeechRecognition || window.webkitSpeechRecognition;

  if (!SpeechRecognition) {
    alert("Speech recognition is not supported in this browser.");
    return;
  }

  const recognition = new SpeechRecognition();
  recognition.lang = "en-US";
  recognition.interimResults = false;
  recognition.maxAlternatives = 1;

  recognition.onresult = (event) => {
    const text = event.results[0][0].transcript;
    google.colab.kernel.invokeFunction(
      "notebook.save_transcript", [text], {}
    );
    document.getElementById("mic-status").innerText =
      "Recognized: " + text;
  };

  recognition.onerror = (event) => {
    document.getElementById("mic-status").innerText =
      "Microphone error: " + event.error;
  };

  recognition.start();
  document.getElementById("mic-status").innerText =
    "Listening... speak now.";
})();
"""))

display(Javascript("""
if (!document.getElementById("mic-status")) {
  const el = document.createElement("p");
  el.id = "mic-status";
  el.innerText = "Preparing microphone...";
  document.body.appendChild(el);
}
"""))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

from IPython.display import display, Javascript
from google.colab import output
import time

transcript_result = {"text": ""}

def save_transcript(text):
    transcript_result["text"] = text

output.register_callback(
    "notebook.save_transcript", save_transcript
)

display(Javascript("""
(async () => {
  const SpeechRecognition =
    window.SpeechRecognition || window.webkitSpeechRecognition;

  if (!SpeechRecognition) {
    alert("Speech recognition is not supported. Try Chrome.");
    return;
  }

  const recognition = new SpeechRecognition();
  recognition.lang = "en-US";
  recognition.interimResults = false;
  recognition.maxAlternatives = 1;

  recognition.onresult = (event) => {
    const text = event.results[0][0].transcript;
    google.colab.kernel.invokeFunction(
      "notebook.save_transcript", [text], {}
    );
  };

  recognition.onerror = (event) => {
    google.colab.kernel.invokeFunction(
      "notebook.save_transcript",
      ["ERROR: " + event.error], {}
    );
  };

  recognition.start();
  alert("Click OK, then speak your sentence.");
})();
"""))

# Wait for speech recognition to finish
for _ in range(60):
    if transcript_result["text"]:
        break
    time.sleep(1)

spoken_sentence = transcript_result["text"]
print("Your spoken sentence:", spoken_sentence)


<IPython.core.display.Javascript object>

Your spoken sentence: 


In [ ]:

from IPython.display import display, Javascript
from google.colab import output

def receive_speech(text):
    print("Your spoken sentence:", text)

output.register_callback(
    "notebook.receive_speech", receive_speech
)

display(Javascript("""
(() => {
  const SR = window.SpeechRecognition ||
             window.webkitSpeechRecognition;

  if (!SR) {
    alert("Speech recognition is not supported in this browser.");
    return;
  }

  const recognition = new SR();
  recognition.lang = "en-US";
  recognition.continuous = false;
  recognition.interimResults = true;

  recognition.onstart = () => {
    console.log("Listening. Speak now!");
  };

  recognition.onresult = (event) => {
    const result = event.results[event.results.length - 1];

    if (result.isFinal) {
      google.colab.kernel.invokeFunction(
        "notebook.receive_speech",
        [result[0].transcript],
        {}
      );
    }
  };

  recognition.onerror = (event) => {
    alert("Speech recognition error: " + event.error);
  };

  recognition.start();
})();
"""))


<IPython.core.display.Javascript object>

Your spoken sentence: the boy is playing in the garden


In [ ]:

# Use the sentence recognized by your microphone
original_message = spoken_sentence
current_message = original_message

print("🎤 Original spoken message:")
print(original_message)
print("\n" + "=" * 45)

random.seed()  # Allow changes to vary between runs

for player in range(1, 9):
    current_message = whisper_change(current_message)
    print(f"Player {player}: {current_message}")

print("\n🏁 Final message:", current_message)


🎤 Original spoken message:


Player 1: 
Player 2: 
Player 3: 
Player 4: 
Player 5: 
Player 6: 
Player 7: 
Player 8: 

🏁 Final message: 


In [ ]:
from IPython.display import display, Javascript
from google.colab import output

spoken_sentence = ""

def save_speech(text):
    global spoken_sentence
    spoken_sentence = text
    print("🎤 Original spoken message:", spoken_sentence)

output.register_callback("notebook.save_speech", save_speech)

display(Javascript("""
(() => {
  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  if (!SR) {
    alert("Speech recognition is not supported.");
    return;
  }

  const recognition = new SR();
  recognition.lang = "en-US";
  recognition.continuous = false;
  recognition.interimResults = false;

  recognition.onresult = (event) => {
    google.colab.kernel.invokeFunction(
      "notebook.save_speech",
      [event.results[0][0].transcript],
      {}
    );
  };

  recognition.onerror = (event) => {
    console.log("Speech error:", event.error);
  };

  recognition.start();
  console.log("Listening... Speak now!");
})();
"""))

<IPython.core.display.Javascript object>

🎤 Original spoken message: the boy is playing in the garden


In [ ]:

from IPython.display import display, Javascript

display(Javascript("""
(async () => {
  try {
    const result = await navigator.permissions.query({
      name: "microphone"
    });
    console.log("Microphone permission:", result.state);

    const stream = await navigator.mediaDevices.getUserMedia({
      audio: true
    });
    console.log("Microphone access: SUCCESS");
    stream.getTracks().forEach(track => track.stop());
  } catch (error) {
    console.log("Error:", error.name, error.message);
  }
})();
"""))


<IPython.core.display.Javascript object>

In [ ]:
original_message = spoken_sentence
current_message = original_message

print("🎤 Original message:", original_message)
print("=" * 45)

for player in range(1, 9):
    current_message = whisper_change(current_message)
    print(f"Player {player}: {current_message}")

print("=" * 45)
print("🏁 Final message:", current_message)

🎤 Original message: the boy is playing in the garden
Player 1: the boy unknown playing in the garden
Player 2: the boy unknown playing the garden
Player 3: the unknown boy playing the garden
Player 4: the unknown playing boy the garden
Player 5: the unknown boy the garden
Player 6: the unknown unknown the garden
Player 7: the unknown unknown the garden
Player 8: the unknown unknown the garden
🏁 Final message: the unknown unknown the garden


In [ ]:

import random

random.seed()

def whisper_change(sentence):
    words = sentence.split()

    if not words:
        return sentence

    change = random.choice(["replace", "remove", "swap"])

    # Replace a word with a similar or commonly confused word
    alternatives = {
        "boy": ["toy", "girl", "man"],
        "girl": ["boy", "world"],
        "playing": ["saying", "staying", "play"],
        "garden": ["gar den", "park", "pardon"],
        "the": ["a", "this"],
        "is": ["was", "his"],
        "in": ["on", "near"],
        "cat": ["cap", "bat"],
        "dog": ["duck", "log"],
        "mat": ["map", "hat"]
    }

    if change == "replace":
        index = random.randrange(len(words))
        word = words[index].lower()

        if word in alternatives:
            words[index] = random.choice(alternatives[word])
        elif len(words) > 2:
            words.pop(index)

    elif change == "remove" and len(words) > 2:
        words.pop(random.randrange(len(words)))

    elif change == "swap" and len(words) > 1:
        index = random.randrange(len(words) - 1)
        words[index], words[index + 1] = (
            words[index + 1], words[index]
        )

    return " ".join(words)


In [ ]:
test_message = "the boy is playing in the garden"

print("Before:", test_message)
print("After: ", whisper_change(test_message))

Before: the boy is playing in the garden
After:  the is playing in the garden


In [ ]:
original_message = spoken_sentence
current_message = original_message

print("🎤 Original message:", original_message)
print("-" * 45)

for player in range(1, 9):
    current_message = whisper_change(current_message)
    print(f"Player {player}: {current_message}")

print("-" * 45)
print("🏁 Final message:", current_message)

🎤 Original message: the boy is playing in the garden
---------------------------------------------
Player 1: the boy is playing on the garden
Player 2: the boy is playing on garden
Player 3: the boy is playing garden
Player 4: the boy is garden
Player 5: the boy is
Player 6: a boy is
Player 7: a boy
Player 8: a girl
---------------------------------------------
🏁 Final message: a girl


In [ ]:

def bilstm_whisper(sentence):
    # Use the trained BiLSTM to predict a changed version
    changed_sentence = predict_sentence(sentence)

    # If the model produces no usable words, keep the message unchanged
    if not changed_sentence.strip():
        return sentence

    return changed_sentence


# Test the BiLSTM on your spoken sentence
print("Original spoken message:", spoken_sentence)
print("BiLSTM output:", bilstm_whisper(spoken_sentence))


Original spoken message: the boy is playing in the garden
BiLSTM output: the cat is playing in the garden


In [ ]:

current_message = spoken_sentence

print("🎤 Original message:", current_message)
print("-" * 50)

for player in range(1, 9):
    predicted = predict_sentence(current_message).strip()

    # Use the prediction if it has enough words
    if len(predicted.split()) >= 3:
        current_message = predicted

    # Each player may also mishear the message
    current_message = whisper_change(current_message)

    print(f"Player {player}: {current_message}")

print("-" * 50)
print("🏁 Final message:", current_message)


🎤 Original message: the boy is playing in the garden
--------------------------------------------------
Player 1: the cat is playing in garden
Player 2: the is playing on the things
Player 3: the is sleeping on trees the things
Player 4: the are are the on the sofa
Player 5: the bat are playing on the sofa
Player 6: the cat playing are on the sofa
Player 7: the cat is on the sofa
Player 8: this cat is sleeping on the things
--------------------------------------------------
🏁 Final message: this cat is sleeping on the things


In [ ]:

original_message = "the boy is playing in the garden"
player_outputs = [
    "the cat is playing in garden",
    "the is playing on the things",
    "the is sleeping on trees the things",
    "the are are the on the sofa",
    "the bat are playing on the sofa",
    "the cat playing are on the sofa",
    "the cat is on the sofa",
    "this cat is sleeping on the things"
]

print("CHINESE WHISPERS USING BiLSTM")
print("Original input:", original_message)

for i, message in enumerate(player_outputs, start=1):
    print(f"Player {i}: {message}")

print("Final output:", player_outputs[-1])


CHINESE WHISPERS USING BiLSTM
Original input: the boy is playing in the garden
Player 1: the cat is playing in garden
Player 2: the is playing on the things
Player 3: the is sleeping on trees the things
Player 4: the are are the on the sofa
Player 5: the bat are playing on the sofa
Player 6: the cat playing are on the sofa
Player 7: the cat is on the sofa
Player 8: this cat is sleeping on the things
Final output: this cat is sleeping on the things
